# Llama 3.2 1B — Specialized TQ-LARC + Dual Score/Output LARC

이번 노트북은 이전 mixed 2/3bit LARC를 버리고, **3bit 전용 LARC**와 **2bit 전용 LARC**를 따로 학습합니다.

핵심 개선:

```text
1. 3bit 전용 보정기와 2bit 전용 보정기를 분리
2. token/group별 fp16 scale 제거
3. layer-wise sigma → layer/head-wise sigma
4. K/V shared codebook → K/V separate learned codebook
5. fixed Gaussian codebook 대신 calibration-based 1D k-means codebook 사용
6. LARC 입력에 quantization code statistics 추가
7. LARC를 Mixture-of-Basis low-rank corrector로 강화
8. 2bit 전용 모델에는 learnable attention score temperature correction 추가
```

비교:

```text
FP baseline

3bit path:
TQ-like K3V3
TQ-like K3V3 + LARC3-only

2bit path:
TQ-like K2V2
TQ-like K2V2 + LARC2-only
```

주의:

```text
공식 TurboQuant 구현이 아니라 TurboQuant-inspired prototype입니다.
현재는 fake quant/dequant입니다.
진짜 compressed-domain attention은 packed int2/int3 + fused kernel이 필요합니다.
```

> This version reports final NLL/PPL and fixed overhead only.


In [ ]:
import os, math, time, random, types, subprocess, sys
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("transformers:", transformers.__version__)
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "sentencepiece"])
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("transformers:", transformers.__version__)

torch.manual_seed(42)
random.seed(42)

In [ ]:
@dataclass
class CFG:
    model_name: str = "meta-llama/Llama-3.2-1B"

    max_len: int = 256
    train_batch_size: int = 2
    eval_batch_size: int = 2
    eval_batches: int = 24

    train_pool_tokens_min: int = 4096
    eval_pool_tokens_min: int = 6144

    # Calibration
    sigma_calib_batches: int = 8
    codebook_calib_batches: int = 6
    max_codebook_samples_per_kind: int = 200_000
    sigma_eps: float = 1e-6
    use_hadamard_rotation: bool = True

    # Specialized training
    train_steps_3bit: int = 180
    train_steps_2bit: int = 220

    # LARC3, smaller because 3bit error is already moderate.
    larc3_feature_dim: int = 128
    larc3_rank: int = 96
    larc3_experts: int = 4
    larc3_hidden_mlp: int = 256

    # LARC2, larger because 2bit error is much harder.
    larc2_feature_dim: int = 160
    larc2_rank: int = 128
    larc2_experts: int = 4
    larc2_hidden_mlp: int = 384

    init_scale: float = 0.01

    # Distillation
    lr: float = 1e-4
    weight_decay: float = 1e-5
    grad_clip: float = 1.0
    kl_topk: int = 256
    kl_temperature: float = 1.0
    kl_weight: float = 1.0
    ce_weight: float = 0.015
    base_no_worse_weight: float = 0.75
    residual_l2_weight: float = 0.02
    gate_l2_weight: float = 0.005
    score_temp_l2_weight: float = 0.005

    amp: bool = True
    dtype: str = "bf16"

cfg = CFG()
print(cfg)

In [ ]:
def get_torch_dtype():
    if cfg.dtype == "bf16" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if cfg.dtype == "fp16" and torch.cuda.is_available():
        return torch.float16
    return torch.float32

torch_dtype = get_torch_dtype()
print("torch_dtype:", torch_dtype)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch_dtype,
    device_map=None,
).to(device)
model.eval()

for p in model.parameters():
    p.requires_grad_(False)

num_layers = len(model.model.layers)
hidden_size = model.config.hidden_size
num_q_heads = model.config.num_attention_heads
num_kv_heads = model.config.num_key_value_heads
head_dim = hidden_size // num_q_heads

print("loaded:", cfg.model_name)
print("num_layers:", num_layers)
print("hidden_size:", hidden_size)
print("num_attention_heads:", num_q_heads)
print("num_key_value_heads:", num_kv_heads)
print("head_dim:", head_dim)

## Dataset

In [ ]:
def load_corpus_texts():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import load_dataset
        ds = load_dataset("wikitext", "wikitext-2-raw-v1")
        train_texts = [x["text"] for x in ds["train"] if len(x["text"].strip()) > 80]
        eval_texts = [x["text"] for x in ds["validation"] if len(x["text"].strip()) > 80]
        print("Loaded Wikitext-2:", len(train_texts), "train texts,", len(eval_texts), "eval texts")
        return train_texts, eval_texts
    except Exception as e:
        print("Wikitext load failed, using fallback corpus:", repr(e))
        base_train = [
            "The history of science is filled with examples of simple ideas becoming powerful tools. "
            "Researchers often begin with a rough approximation and then improve it through careful measurement.",
            "Artificial intelligence systems use patterns in data to predict the next word in a sequence. "
            "A reliable experiment compares the original model, the compressed model, and the corrected model.",
            "Language models store key and value tensors during generation so that previous tokens can be reused efficiently. "
            "A small error in one layer may be amplified by later layers.",
            "The engineer tested the model carefully, comparing the original output with the compressed output. "
            "The result was not accepted until it worked on a separate evaluation set.",
        ] * 256
        base_eval = [
            "During inference, every layer contributes to the final prediction, so small errors can accumulate across the residual stream. "
            "A correction module should improve the final token distribution, not only the local vector similarity.",
            "A good compression method should reduce memory while preserving the information needed for accurate reasoning. "
            "Evaluation must use text that was not used for training.",
            "When the sun went down, the city lights appeared one by one across the river. "
            "The story continued with details that had not appeared in the training examples.",
            "The researcher decided to remove the dimension reduction step and test pure low-bit quantization instead. "
            "The correction network was judged by its performance on held-out text.",
        ] * 256
        return base_train, base_eval

train_texts_raw, eval_texts_raw = load_corpus_texts()

def tokenize_texts_to_ids(texts, min_tokens):
    ids_list = []
    total = 0
    for text in texts:
        toks = tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids[0]
        if toks.numel() < 8:
            continue
        ids_list.append(toks)
        total += toks.numel()
        if total >= min_tokens:
            break
    if not ids_list:
        raise RuntimeError("No tokenized texts available.")
    return torch.cat(ids_list, dim=0)

train_ids_cpu = tokenize_texts_to_ids(train_texts_raw, cfg.train_pool_tokens_min)
eval_ids_cpu = tokenize_texts_to_ids(eval_texts_raw, cfg.eval_pool_tokens_min)

print("train tokens:", train_ids_cpu.numel())
print("eval tokens:", eval_ids_cpu.numel())

def make_batch_from_ids(ids_cpu, batch_size, max_len, random_sample=True, batch_index=0):
    ids_cpu = ids_cpu.cpu()
    need = max_len + 1
    max_start = ids_cpu.numel() - need
    if max_start <= 0:
        raise ValueError("Not enough tokens for requested max_len.")
    chunks = []
    if random_sample:
        starts = torch.randint(0, max_start, (batch_size,))
    else:
        stride = need
        starts = []
        for b in range(batch_size):
            s = ((batch_index * batch_size + b) * stride) % max_start
            starts.append(s)
        starts = torch.tensor(starts)
    for s in starts:
        chunks.append(ids_cpu[int(s): int(s) + need])
    x = torch.stack(chunks, dim=0)
    input_ids = x[:, :-1].contiguous().to(device)
    labels_ref = x[:, 1:].contiguous().to(device)
    attention_mask = torch.ones_like(input_ids, device=device)
    return {"input_ids": input_ids, "attention_mask": attention_mask}, labels_ref

train_batch, train_labels_ref = make_batch_from_ids(train_ids_cpu, cfg.train_batch_size, cfg.max_len, True)
eval_batch0, eval_labels0 = make_batch_from_ids(eval_ids_cpu, cfg.eval_batch_size, cfg.max_len, False, 0)

print("train batch:", train_batch["input_ids"].shape)
print("eval batch:", eval_batch0["input_ids"].shape)

## Hadamard rotation utility

In [ ]:
def fwht_ortho(x):
    # x shape: [..., D], D must be power of two
    D = x.shape[-1]
    assert D & (D - 1) == 0, "head_dim must be power of two for Hadamard rotation"
    y = x.float()
    h = 1
    while h < D:
        y = y.reshape(*y.shape[:-1], -1, h * 2)
        a = y[..., :, :h]
        b = y[..., :, h:]
        y = torch.cat([a + b, a - b], dim=-1)
        y = y.reshape(*y.shape[:-2], -1)
        h *= 2
    return (y / math.sqrt(D)).to(dtype=x.dtype)

## Calibration 1: layer/head-wise fixed sigma

이전 버전은 `sigma[layer]`였지만, 이번에는 `sigma[layer, kv_head]`를 사용합니다.

오버헤드:

```text
16 layers × 8 kv_heads × 2(K,V) fp16 ≈ 512 bytes
```

In [ ]:
@torch.no_grad()
def calibrate_headwise_kv_sigma(n_batches=8):
    sums_k = torch.zeros(num_layers, num_kv_heads, dtype=torch.float64)
    sums_v = torch.zeros(num_layers, num_kv_heads, dtype=torch.float64)
    counts_k = torch.zeros(num_layers, num_kv_heads, dtype=torch.float64)
    counts_v = torch.zeros(num_layers, num_kv_heads, dtype=torch.float64)

    handles = []

    def make_hook(layer_idx):
        def hook(module, args, kwargs):
            if "hidden_states" in kwargs:
                h = kwargs["hidden_states"]
            elif len(args) > 0:
                h = args[0]
            else:
                return

            B, T, C = h.shape
            D = C // model.config.num_attention_heads
            Hkv = model.config.num_key_value_heads

            k = module.k_proj(h).float().view(B, T, Hkv, D).transpose(1, 2)  # [B,H,T,D]
            v = module.v_proj(h).float().view(B, T, Hkv, D).transpose(1, 2)

            sums_k[layer_idx] += (k ** 2).sum(dim=(0, 2, 3)).detach().cpu().double()
            sums_v[layer_idx] += (v ** 2).sum(dim=(0, 2, 3)).detach().cpu().double()
            counts_k[layer_idx] += k.shape[0] * k.shape[2] * k.shape[3]
            counts_v[layer_idx] += v.shape[0] * v.shape[2] * v.shape[3]

        return hook

    for li, layer in enumerate(model.model.layers):
        handles.append(layer.self_attn.register_forward_pre_hook(make_hook(li), with_kwargs=True))

    for bi in range(n_batches):
        batch, _ = make_batch_from_ids(train_ids_cpu, cfg.train_batch_size, cfg.max_len, random_sample=True)
        _ = model(**batch, use_cache=False)

    for h in handles:
        h.remove()

    sigma_k = torch.sqrt((sums_k / counts_k).clamp_min(cfg.sigma_eps)).float().to(device)
    sigma_v = torch.sqrt((sums_v / counts_v).clamp_min(cfg.sigma_eps)).float().to(device)
    return sigma_k, sigma_v

sigma_k_lh, sigma_v_lh = calibrate_headwise_kv_sigma(n_batches=cfg.sigma_calib_batches)

print("sigma_k_lh shape:", sigma_k_lh.shape)
print("sigma_v_lh shape:", sigma_v_lh.shape)
print("headwise fixed sigma metadata fp16 MB:", (sigma_k_lh.numel() + sigma_v_lh.numel()) * 2 / 1024**2)
print("sigma_k first layer:", sigma_k_lh[0].detach().cpu().numpy())
print("sigma_v first layer:", sigma_v_lh[0].detach().cpu().numpy())

## Calibration 2: K/V separate learned codebooks

이전 버전은 고정 Gaussian codebook이었습니다.  
이번에는 calibration sample을 모아서 K/V별로 1D k-means codebook을 학습합니다.

```text
codebook_K_3bit
codebook_V_3bit
codebook_K_2bit
codebook_V_2bit
```

이 codebook은 고정 overhead이며 token 수에 따라 증가하지 않습니다.

In [ ]:
@torch.no_grad()
def collect_normalized_rotated_samples(n_batches=6, max_samples_per_kind=200_000):
    samples = {"k": [], "v": []}
    per_hook_limit = max(512, max_samples_per_kind // max(1, n_batches * num_layers))

    handles = []

    def maybe_sample(flat):
        flat = flat.detach().float().flatten().cpu()
        if flat.numel() > per_hook_limit:
            idx = torch.randperm(flat.numel())[:per_hook_limit]
            flat = flat[idx]
        return flat

    def make_hook(layer_idx):
        def hook(module, args, kwargs):
            if "hidden_states" in kwargs:
                h = kwargs["hidden_states"]
            elif len(args) > 0:
                h = args[0]
            else:
                return

            B, T, C = h.shape
            D = C // model.config.num_attention_heads
            Hkv = model.config.num_key_value_heads

            k = module.k_proj(h).float().view(B, T, Hkv, D).transpose(1, 2)  # [B,H,T,D]
            v = module.v_proj(h).float().view(B, T, Hkv, D).transpose(1, 2)

            sk = sigma_k_lh[layer_idx].view(1, Hkv, 1, 1).to(k.device).float().clamp_min(cfg.sigma_eps)
            sv = sigma_v_lh[layer_idx].view(1, Hkv, 1, 1).to(v.device).float().clamp_min(cfg.sigma_eps)

            kn = k / sk
            vn = v / sv

            if cfg.use_hadamard_rotation:
                kn = fwht_ortho(kn)
                vn = fwht_ortho(vn)

            samples["k"].append(maybe_sample(kn))
            samples["v"].append(maybe_sample(vn))

        return hook

    for li, layer in enumerate(model.model.layers):
        handles.append(layer.self_attn.register_forward_pre_hook(make_hook(li), with_kwargs=True))

    for bi in range(n_batches):
        batch, _ = make_batch_from_ids(train_ids_cpu, cfg.train_batch_size, cfg.max_len, random_sample=True)
        _ = model(**batch, use_cache=False)

    for h in handles:
        h.remove()

    out = {}
    for kind in ["k", "v"]:
        x = torch.cat(samples[kind], dim=0)
        if x.numel() > max_samples_per_kind:
            idx = torch.randperm(x.numel())[:max_samples_per_kind]
            x = x[idx]
        out[kind] = x
        print(kind, "samples:", x.shape, "mean", float(x.mean()), "std", float(x.std()))

    return out

def kmeans_1d(samples, n_levels, iters=30):
    x = samples.float().cpu()
    # robust clipping for stable centroids
    lo = torch.quantile(x, 0.001)
    hi = torch.quantile(x, 0.999)
    x = x.clamp(lo, hi)

    qs = torch.linspace(0.0, 1.0, n_levels + 2)[1:-1]
    centroids = torch.quantile(x, qs).float()

    for _ in range(iters):
        dist = (x[:, None] - centroids[None, :]) ** 2
        idx = dist.argmin(dim=1)
        new = []
        for j in range(n_levels):
            m = x[idx == j]
            if m.numel() == 0:
                new.append(centroids[j])
            else:
                new.append(m.mean())
        centroids = torch.stack(new).float()
        centroids, _ = torch.sort(centroids)

    return centroids

calib_samples = collect_normalized_rotated_samples(
    n_batches=cfg.codebook_calib_batches,
    max_samples_per_kind=cfg.max_codebook_samples_per_kind,
)

CODEBOOKS = {}
for kind in ["k", "v"]:
    for bits in [2, 3]:
        cb = kmeans_1d(calib_samples[kind], n_levels=2**bits, iters=30)
        CODEBOOKS[(kind, bits)] = cb.to(device)
        print(f"CODEBOOK {kind} {bits}bit:", cb.numpy())

codebook_overhead_MB = sum(cb.numel() for cb in CODEBOOKS.values()) * 2 / 1024**2
print("learned codebook fp16 overhead MB:", codebook_overhead_MB)

## Quantization functions

In [ ]:
def nearest_codebook_quantize(y, kind, bits):
    codebook = CODEBOOKS[(kind, int(bits))].to(device=y.device, dtype=y.dtype)
    dist = (y.unsqueeze(-1).float() - codebook.float().view(*([1] * y.ndim), -1)) ** 2
    idx = dist.argmin(dim=-1)
    y_hat = codebook[idx].to(dtype=y.dtype)
    return y_hat, idx

def tq_quantize_dequant_with_stats(x, layer_idx, kind, bits):
    """
    x: [B,Hkv,T,D]
    Returns:
      x_hat: same shape
      stats_abs: [B,T]
      stats_sat: [B,T]
      stats_err: [B,T]
    """
    Hkv = x.shape[1]
    if kind == "k":
        sigma = sigma_k_lh[int(layer_idx)].view(1, Hkv, 1, 1).to(device=x.device, dtype=x.dtype)
    else:
        sigma = sigma_v_lh[int(layer_idx)].view(1, Hkv, 1, 1).to(device=x.device, dtype=x.dtype)

    y = x / sigma.clamp_min(cfg.sigma_eps)

    if cfg.use_hadamard_rotation:
        y_rot = fwht_ortho(y)
    else:
        y_rot = y

    yq_rot, idx = nearest_codebook_quantize(y_rot, kind=kind, bits=bits)

    if cfg.use_hadamard_rotation:
        yq = fwht_ortho(yq_rot)
    else:
        yq = yq_rot

    x_hat = (yq * sigma).to(dtype=x.dtype)

    # Code statistics. Shape [B,H,T,D] -> [B,T]
    n_levels = 2 ** int(bits)
    idx_f = idx.float()
    centered = (idx_f - (n_levels - 1) / 2.0).abs() / max((n_levels - 1) / 2.0, 1.0)
    stat_abs = centered.mean(dim=(1, 3))
    stat_sat = ((idx == 0) | (idx == n_levels - 1)).float().mean(dim=(1, 3))
    stat_err = ((y_rot.float() - yq_rot.float()) ** 2).mean(dim=(1, 3)).sqrt()

    return x_hat, stat_abs, stat_sat, stat_err

def kv_payload_effective_bits(k_bits, v_bits):
    return (float(k_bits) + float(v_bits)) / 2.0

print("K3V3 payload bits:", kv_payload_effective_bits(3, 3), "compression:", 16 / kv_payload_effective_bits(3, 3))
print("K2V2 payload bits:", kv_payload_effective_bits(2, 2), "compression:", 16 / kv_payload_effective_bits(2, 2))
print("K3V2 payload bits:", kv_payload_effective_bits(3, 2), "compression:", 16 / kv_payload_effective_bits(3, 2))

## Mixture-of-Basis LARC

In [ ]:
class MixtureOfBasisLARC(nn.Module):
    """
    Shared across all layers, but layer embedding and learned score temperature are included.
    Correction:
        delta = sum_e gate_e * (coeff_e @ basis_e)
    """
    def __init__(
        self,
        num_layers,
        num_heads,
        hidden_size,
        stat_dim,
        feature_dim=128,
        rank=96,
        experts=4,
        mlp_hidden=256,
        init_scale=0.01,
        use_score_temp=True,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.hidden_size = hidden_size
        self.rank = rank
        self.experts = experts
        self.use_score_temp = use_score_temp

        self.hidden_proj = nn.Linear(hidden_size, feature_dim, bias=False)
        self.out_proj = nn.Linear(hidden_size, feature_dim, bias=False)
        self.stat_proj = nn.Linear(stat_dim, feature_dim, bias=True)
        self.layer_emb = nn.Embedding(num_layers, feature_dim)

        self.norm = nn.LayerNorm(feature_dim)
        self.coeff_net = nn.Sequential(
            nn.Linear(feature_dim, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, experts * rank),
        )
        self.expert_gate = nn.Sequential(
            nn.Linear(feature_dim, mlp_hidden // 2),
            nn.GELU(),
            nn.Linear(mlp_hidden // 2, experts),
        )
        self.token_gate = nn.Sequential(
            nn.Linear(feature_dim, mlp_hidden // 2),
            nn.GELU(),
            nn.Linear(mlp_hidden // 2, 1),
        )

        self.basis = nn.Parameter(torch.randn(experts, rank, hidden_size) * 0.01)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(init_scale / (1.0 - init_scale))))

        if use_score_temp:
            self.score_log_temp = nn.Parameter(torch.zeros(num_layers, num_heads))
        else:
            self.register_parameter("score_log_temp", None)

        # Start as near no-op.
        nn.init.zeros_(self.coeff_net[-1].weight)
        nn.init.zeros_(self.coeff_net[-1].bias)
        nn.init.zeros_(self.expert_gate[-1].weight)
        nn.init.zeros_(self.expert_gate[-1].bias)
        nn.init.zeros_(self.token_gate[-1].weight)
        nn.init.constant_(self.token_gate[-1].bias, -2.0)

    def score_temperature(self, layer_idx):
        if self.score_log_temp is None:
            return None
        # Clamp to avoid early instability.
        return torch.exp(self.score_log_temp[int(layer_idx)].clamp(-0.25, 0.25))

    def forward_layer(self, layer_idx, hidden_states, base_out, quant_stats):
        B, T, C = hidden_states.shape
        layer_ids = torch.full((B, T), int(layer_idx), device=hidden_states.device, dtype=torch.long)

        feat = (
            self.hidden_proj(hidden_states)
            + self.out_proj(base_out)
            + self.stat_proj(quant_stats.to(base_out.dtype))
            + self.layer_emb(layer_ids).to(base_out.dtype)
        )
        feat = self.norm(feat)

        coeff = self.coeff_net(feat).view(B, T, self.experts, self.rank)       # [B,T,E,R]
        egate = F.softmax(self.expert_gate(feat).float(), dim=-1).to(feat.dtype) # [B,T,E]
        delta_e = torch.einsum("bter,erh->bteh", coeff, self.basis.to(coeff.dtype))
        delta = (delta_e * egate.unsqueeze(-1)).sum(dim=2)

        tgate = torch.sigmoid(self.token_gate(feat))
        gscale = torch.sigmoid(self.logit_scale)
        correction = gscale * tgate * delta

        return base_out + correction, correction, tgate.mean()

# stats:
# log_sigma_k_mean, log_sigma_v_mean,
# k_code_abs, k_sat, k_err,
# v_code_abs, v_sat, v_err,
# relative_depth, one
STAT_DIM = 10

larc3 = MixtureOfBasisLARC(
    num_layers=num_layers,
    num_heads=num_q_heads,
    hidden_size=hidden_size,
    stat_dim=STAT_DIM,
    feature_dim=cfg.larc3_feature_dim,
    rank=cfg.larc3_rank,
    experts=cfg.larc3_experts,
    mlp_hidden=cfg.larc3_hidden_mlp,
    init_scale=cfg.init_scale,
    use_score_temp=False,
).to(device=device, dtype=torch_dtype)

larc2 = MixtureOfBasisLARC(
    num_layers=num_layers,
    num_heads=num_q_heads,
    hidden_size=hidden_size,
    stat_dim=STAT_DIM,
    feature_dim=cfg.larc2_feature_dim,
    rank=cfg.larc2_rank,
    experts=cfg.larc2_experts,
    mlp_hidden=cfg.larc2_hidden_mlp,
    init_scale=cfg.init_scale,
    use_score_temp=True,
).to(device=device, dtype=torch_dtype)

LARC_CONTROLLERS = {
    "3bit": larc3,
    "2bit": larc2,
}

def param_count(m):
    return sum(p.numel() for p in m.parameters())

print("LARC3 params:", param_count(larc3), "fp16 MB:", param_count(larc3) * 2 / 1024**2)
print("LARC2 params:", param_count(larc2), "fp16 MB:", param_count(larc2) * 2 / 1024**2)
print("Total LARC params:", param_count(larc3) + param_count(larc2))

## Patch Llama attention

In [ ]:
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv

LARC_STATE = {
    "quant_enabled": False,
    "larc_enabled": False,
    "k_bits": 3,
    "v_bits": 3,
    "active_larc": "3bit",
    "collect_stats": False,
    "gate_means": [],
    "corr_norms": [],
}

_ORIGINAL_FORWARDS = {}

def _get_layer_idx(attn):
    if hasattr(attn, "layer_idx") and attn.layer_idx is not None:
        return int(attn.layer_idx)
    return int(getattr(attn, "_larc_layer_idx"))

def make_quant_stats(layer_idx, bsz, q_len, k_stats, v_stats, dtype, device):
    k_abs, k_sat, k_err = k_stats
    v_abs, v_sat, v_err = v_stats

    sk_log = sigma_k_lh[int(layer_idx)].mean().to(device=device).float().log().expand(bsz, q_len)
    sv_log = sigma_v_lh[int(layer_idx)].mean().to(device=device).float().log().expand(bsz, q_len)
    depth = torch.full((bsz, q_len), float(layer_idx) / max(1, num_layers - 1), device=device)
    ones = torch.ones((bsz, q_len), device=device)

    stats = torch.stack([
        sk_log,
        sv_log,
        k_abs.float(),
        k_sat.float(),
        k_err.float(),
        v_abs.float(),
        v_sat.float(),
        v_err.float(),
        depth,
        ones,
    ], dim=-1)

    return stats.to(dtype=dtype)

def make_larc_attention_forward(attn_module):
    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        **kwargs,
    ):
        bsz, q_len, _ = hidden_states.size()
        layer_idx = _get_layer_idx(self)
        k_bits = int(LARC_STATE["k_bits"])
        v_bits = int(LARC_STATE["v_bits"])

        num_heads = self.config.num_attention_heads
        num_kv_heads_local = self.config.num_key_value_heads
        head_dim_local = self.config.hidden_size // self.config.num_attention_heads

        query_states = self.q_proj(hidden_states).view(bsz, q_len, num_heads, head_dim_local).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(bsz, q_len, num_kv_heads_local, head_dim_local).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(bsz, q_len, num_kv_heads_local, head_dim_local).transpose(1, 2)

        if position_embeddings is None:
            if position_ids is None:
                position_ids = torch.arange(q_len, device=hidden_states.device).unsqueeze(0)
            cos, sin = self.rotary_emb(value_states, position_ids)
        else:
            cos, sin = position_embeddings

        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            key_states, value_states = past_key_value.update(key_states, value_states, layer_idx, cache_kwargs)

        if LARC_STATE["quant_enabled"]:
            key_states, k_abs, k_sat, k_err = tq_quantize_dequant_with_stats(
                key_states, layer_idx, "k", bits=k_bits
            )
            value_states, v_abs, v_sat, v_err = tq_quantize_dequant_with_stats(
                value_states, layer_idx, "v", bits=v_bits
            )
            quant_stats = make_quant_stats(
                layer_idx, bsz, q_len,
                (k_abs, k_sat, k_err),
                (v_abs, v_sat, v_err),
                hidden_states.dtype,
                hidden_states.device,
            )
        else:
            quant_stats = torch.zeros(bsz, q_len, STAT_DIM, device=hidden_states.device, dtype=hidden_states.dtype)

        key_states_rep = repeat_kv(key_states, self.num_key_value_groups)
        value_states_rep = repeat_kv(value_states, self.num_key_value_groups)

        attn_scores = torch.matmul(query_states, key_states_rep.transpose(2, 3)) / math.sqrt(head_dim_local)

        # 2bit-specific score temperature correction.
        if LARC_STATE["quant_enabled"] and LARC_STATE["larc_enabled"]:
            ctrl = LARC_CONTROLLERS[LARC_STATE["active_larc"]]
            temp = ctrl.score_temperature(layer_idx) if hasattr(ctrl, "score_temperature") else None
            if temp is not None:
                attn_scores = attn_scores * temp.view(1, -1, 1, 1).to(attn_scores.dtype)

        if attention_mask is not None:
            causal_mask = attention_mask[:, :, :, : key_states_rep.shape[-2]]
            attn_scores = attn_scores + causal_mask

        attn_weights = F.softmax(attn_scores.float(), dim=-1).to(query_states.dtype)
        attn_weights = F.dropout(attn_weights, p=self.attention_dropout, training=self.training)
        attn_output = torch.matmul(attn_weights, value_states_rep)

        attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, -1)
        base_out = self.o_proj(attn_output)

        if LARC_STATE["quant_enabled"] and LARC_STATE["larc_enabled"]:
            ctrl = LARC_CONTROLLERS[LARC_STATE["active_larc"]]
            corrected, corr, gate_mean = ctrl.forward_layer(layer_idx, hidden_states, base_out, quant_stats)

            if LARC_STATE["collect_stats"]:
                LARC_STATE["gate_means"].append(float(gate_mean.detach().float().cpu()))
                LARC_STATE["corr_norms"].append(float((corr.float().norm() / (base_out.float().norm() + 1e-8)).detach().cpu()))

            attn_output_final = corrected
        else:
            attn_output_final = base_out

        if not output_attentions:
            attn_weights = None

        return attn_output_final, attn_weights

    return forward

def patch_all_llama_attentions():
    for i, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        attn._larc_layer_idx = i
        if i not in _ORIGINAL_FORWARDS:
            _ORIGINAL_FORWARDS[i] = attn.forward
        attn.forward = types.MethodType(make_larc_attention_forward(attn), attn)
    print(f"Patched all {len(model.model.layers)} attention layers.")

patch_all_llama_attentions()

with torch.no_grad():
    LARC_STATE["quant_enabled"] = False
    LARC_STATE["larc_enabled"] = False
    y = model(**eval_batch0, use_cache=False).logits
print("smoke logits:", y.shape)

## Evaluation utilities

In [ ]:
@torch.no_grad()
def compute_nll_ppl_from_logits_and_labels(logits, labels_ref, attention_mask=None):
    logits = logits.contiguous().float()
    labels_ref = labels_ref.contiguous()
    if attention_mask is not None:
        mask = attention_mask.contiguous().bool()
    else:
        mask = torch.ones_like(labels_ref, dtype=torch.bool)

    vocab = logits.size(-1)
    loss_flat = F.cross_entropy(
        logits.reshape(-1, vocab),
        labels_ref.reshape(-1),
        reduction="none",
    ).view_as(labels_ref)
    loss_sum = loss_flat[mask].sum().item()
    ntok = int(mask.sum().item())
    return loss_sum, ntok

@torch.no_grad()
def evaluate_n_batches(label, quant=False, larc=False, k_bits=3, v_bits=3, active_larc="3bit", n_batches=None):
    if n_batches is None:
        n_batches = cfg.eval_batches

    LARC_STATE["quant_enabled"] = quant
    LARC_STATE["larc_enabled"] = larc
    LARC_STATE["k_bits"] = int(k_bits)
    LARC_STATE["v_bits"] = int(v_bits)
    LARC_STATE["active_larc"] = active_larc
    LARC_STATE["collect_stats"] = False

    total_loss = 0.0
    total_tokens = 0
    elapsed = 0.0

    for bi in range(n_batches):
        batch, labels_ref = make_batch_from_ids(
            eval_ids_cpu,
            cfg.eval_batch_size,
            cfg.max_len,
            random_sample=False,
            batch_index=bi,
        )
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model(**batch, use_cache=False)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed += time.perf_counter() - t0

        loss_sum, ntok = compute_nll_ppl_from_logits_and_labels(
            out.logits,
            labels_ref,
            batch.get("attention_mask"),
        )
        total_loss += loss_sum
        total_tokens += ntok

    nll = total_loss / max(total_tokens, 1)
    ppl = math.exp(nll)
    return {
        "label": label,
        "K_bits": k_bits if quant else 16,
        "V_bits": v_bits if quant else 16,
        "payload_eff_bits": kv_payload_effective_bits(k_bits, v_bits) if quant else 16.0,
        "nll": nll,
        "ppl": ppl,
        "loss_tokens": total_tokens,
        "elapsed_s": elapsed,
        "tokens_per_s": total_tokens / max(elapsed, 1e-9),
    }

## Baseline before LARC training

In [ ]:
import pandas as pd

base_rows = []
base_rows.append(evaluate_n_batches("fp_baseline", quant=False, larc=False, n_batches=cfg.eval_batches))
base_rows.append(evaluate_n_batches("tq_K3V3_base", quant=True, larc=False, k_bits=3, v_bits=3, n_batches=cfg.eval_batches))
base_rows.append(evaluate_n_batches("tq_K2V2_base", quant=True, larc=False, k_bits=2, v_bits=2, n_batches=cfg.eval_batches))
base_rows.append(evaluate_n_batches("tq_K3V2_base_optional", quant=True, larc=False, k_bits=3, v_bits=2, n_batches=cfg.eval_batches))

base_eval_df = pd.DataFrame(base_rows)
display(base_eval_df)

print("Fixed sigma MB:", (sigma_k_lh.numel() + sigma_v_lh.numel()) * 2 / 1024**2)
print("Learned codebook MB:", codebook_overhead_MB)
print("LARC3 fp16 MB:", param_count(larc3) * 2 / 1024**2)
print("LARC2 fp16 MB:", param_count(larc2) * 2 / 1024**2)

## Specialized training functions

In [ ]:
def autocast_ctx():
    if device == "cuda":
        dtype = torch.bfloat16 if torch_dtype == torch.bfloat16 else torch.float16
        return torch.amp.autocast("cuda", enabled=cfg.amp, dtype=dtype)
    return torch.amp.autocast("cpu", enabled=False)

def teacher_topk(logits, topk, temperature=1.0):
    vals, idx = torch.topk(logits.float(), k=topk, dim=-1)
    log_probs = F.log_softmax(vals / temperature, dim=-1)
    probs = log_probs.exp()
    return idx, probs, log_probs

def masked_topk_kl_from_teacher(student_logits, top_idx, teacher_probs, teacher_log_probs, mask, temperature=1.0):
    student_log_probs_full = F.log_softmax(student_logits.float() / temperature, dim=-1)
    student_log_probs_topk = torch.gather(student_log_probs_full, dim=-1, index=top_idx)
    kl = (teacher_probs * (teacher_log_probs - student_log_probs_topk)).sum(dim=-1)
    return kl[mask].mean()

def masked_ce_from_logits(student_logits, labels, mask):
    vocab = student_logits.size(-1)
    ce = F.cross_entropy(student_logits.float().reshape(-1, vocab), labels.reshape(-1), reduction="none").view_as(labels)
    return ce[mask].mean()

def score_temp_l2(controller):
    if getattr(controller, "score_log_temp", None) is None:
        return torch.tensor(0.0, device=device)
    return (controller.score_log_temp.float() ** 2).mean()

def train_specialized_larc(
    name,
    controller,
    k_bits,
    v_bits,
    steps,
    lr=None,
):
    if lr is None:
        lr = cfg.lr

    optimizer = torch.optim.AdamW(controller.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    history = []
    t0 = time.perf_counter()

    for step in range(1, steps + 1):
        train_batch, labels_ref = make_batch_from_ids(
            train_ids_cpu,
            cfg.train_batch_size,
            cfg.max_len,
            random_sample=True,
        )
        mask = train_batch["attention_mask"].bool()

        with torch.no_grad():
            LARC_STATE["quant_enabled"] = False
            LARC_STATE["larc_enabled"] = False
            teacher_logits = model(**train_batch, use_cache=False).logits.detach()
            top_idx, teacher_probs, teacher_log_probs = teacher_topk(
                teacher_logits,
                cfg.kl_topk,
                cfg.kl_temperature,
            )

            LARC_STATE["quant_enabled"] = True
            LARC_STATE["larc_enabled"] = False
            LARC_STATE["k_bits"] = int(k_bits)
            LARC_STATE["v_bits"] = int(v_bits)
            LARC_STATE["active_larc"] = name
            base_quant_logits = model(**train_batch, use_cache=False).logits.detach()
            base_kl = masked_topk_kl_from_teacher(
                base_quant_logits,
                top_idx,
                teacher_probs,
                teacher_log_probs,
                mask,
                cfg.kl_temperature,
            ).detach()

        optimizer.zero_grad(set_to_none=True)

        LARC_STATE["quant_enabled"] = True
        LARC_STATE["larc_enabled"] = True
        LARC_STATE["k_bits"] = int(k_bits)
        LARC_STATE["v_bits"] = int(v_bits)
        LARC_STATE["active_larc"] = name
        LARC_STATE["collect_stats"] = True
        LARC_STATE["gate_means"] = []
        LARC_STATE["corr_norms"] = []

        with autocast_ctx():
            student_logits = model(**train_batch, use_cache=False).logits
            kl = masked_topk_kl_from_teacher(
                student_logits,
                top_idx,
                teacher_probs,
                teacher_log_probs,
                mask,
                cfg.kl_temperature,
            )
            ce = masked_ce_from_logits(student_logits, labels_ref, mask)
            no_worse = F.relu(kl - base_kl)

            gate_mean = torch.tensor(sum(LARC_STATE["gate_means"]) / max(1, len(LARC_STATE["gate_means"])), device=kl.device)
            corr_norm = torch.tensor(sum(LARC_STATE["corr_norms"]) / max(1, len(LARC_STATE["corr_norms"])), device=kl.device)

            loss = (
                cfg.kl_weight * kl
                + cfg.ce_weight * ce
                + cfg.base_no_worse_weight * no_worse
                + cfg.residual_l2_weight * corr_norm
                + cfg.gate_l2_weight * gate_mean
                + cfg.score_temp_l2_weight * score_temp_l2(controller)
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(controller.parameters(), cfg.grad_clip)
        optimizer.step()

        if step == 1 or step % 10 == 0 or step == steps:
            row = {
                "name": name,
                "step": step,
                "K_bits": k_bits,
                "V_bits": v_bits,
                "payload_bits": kv_payload_effective_bits(k_bits, v_bits),
                "loss": float(loss.detach().cpu()),
                "kl_topk": float(kl.detach().cpu()),
                "base_kl_topk": float(base_kl.detach().cpu()),
                "ce_student": float(ce.detach().cpu()),
                "gate_mean": float(gate_mean.detach().cpu()),
                "corr_norm_ratio": float(corr_norm.detach().cpu()),
                "score_temp_l2": float(score_temp_l2(controller).detach().cpu()),
                "elapsed_s": time.perf_counter() - t0,
            }
            history.append(row)
            print(row)

    return pd.DataFrame(history)

## Train 3bit-only LARC

In [ ]:
train3_hist_df = train_specialized_larc(
    name="3bit",
    controller=larc3,
    k_bits=3,
    v_bits=3,
    steps=cfg.train_steps_3bit,
    lr=cfg.lr,
)
display(train3_hist_df)

## Train 2bit-only LARC

In [ ]:
train2_hist_df = train_specialized_larc(
    name="2bit",
    controller=larc2,
    k_bits=2,
    v_bits=2,
    steps=cfg.train_steps_2bit,
    lr=cfg.lr,
)
display(train2_hist_df)

## Final held-out evaluation

In [ ]:
final_rows = []
final_rows.append(evaluate_n_batches("fp_baseline", quant=False, larc=False, n_batches=cfg.eval_batches))

# 3bit path
final_rows.append(evaluate_n_batches("tq_K3V3_base", quant=True, larc=False, k_bits=3, v_bits=3, n_batches=cfg.eval_batches))
final_rows.append(evaluate_n_batches("tq_K3V3_LARC3_only", quant=True, larc=True, k_bits=3, v_bits=3, active_larc="3bit", n_batches=cfg.eval_batches))

# 2bit path
final_rows.append(evaluate_n_batches("tq_K2V2_base", quant=True, larc=False, k_bits=2, v_bits=2, n_batches=cfg.eval_batches))
final_rows.append(evaluate_n_batches("tq_K2V2_LARC2_only", quant=True, larc=True, k_bits=2, v_bits=2, active_larc="2bit", n_batches=cfg.eval_batches))

# Optional 2.5bit diagnostic using 2bit LARC
final_rows.append(evaluate_n_batches("tq_K3V2_base_optional", quant=True, larc=False, k_bits=3, v_bits=2, n_batches=cfg.eval_batches))
final_rows.append(evaluate_n_batches("tq_K3V2_LARC2_optional", quant=True, larc=True, k_bits=3, v_bits=2, active_larc="2bit", n_batches=cfg.eval_batches))

final_df = pd.DataFrame(final_rows)
display(final_df)

print("="*80)
print("FINAL SPECIALIZED REPORT")
print("="*80)

fp = final_df[final_df["label"] == "fp_baseline"].iloc[0]
print(f"FP baseline PPL: {fp['ppl']:.6f}, NLL: {fp['nll']:.6f}")
print("")

def report_pair(base_label, larc_label):
    base = final_df[final_df["label"] == base_label].iloc[0]
    larc = final_df[final_df["label"] == larc_label].iloc[0]
    print(f"[{larc_label}]")
    print(f"payload bits        : {larc['payload_eff_bits']:.3f}")
    print(f"base PPL            : {base['ppl']:.6f}")
    print(f"LARC PPL            : {larc['ppl']:.6f}")
    print(f"PPL improvement     : {base['ppl'] - larc['ppl']:+.6f}")
    print(f"base NLL            : {base['nll']:.6f}")
    print(f"LARC NLL            : {larc['nll']:.6f}")
    print(f"NLL improvement     : {base['nll'] - larc['nll']:+.6f}")
    print(f"compression vs fp16 : {16 / larc['payload_eff_bits']:.3f}x")
    if larc["ppl"] < base["ppl"]:
        print("✅ SUCCESS: LARC improved quantized baseline.")
    else:
        print("❌ FAIL: LARC did not improve quantized baseline.")
    print("")

report_pair("tq_K3V3_base", "tq_K3V3_LARC3_only")
report_pair("tq_K2V2_base", "tq_K2V2_LARC2_only")
report_pair("tq_K3V2_base_optional", "tq_K3V2_LARC2_optional")

## Effective bits and fixed overhead

In [ ]:
def specialized_effective_bits_report(
    model,
    context_lengths=(512, 1024, 2048, 4096, 8192, 16384, 32768),
    batch_size=1,
):
    L = len(model.model.layers)
    Hq = model.config.num_attention_heads
    Hkv = model.config.num_key_value_heads
    hidden = model.config.hidden_size
    D = hidden // Hq

    sigma_MB = (sigma_k_lh.numel() + sigma_v_lh.numel()) * 2 / 1024**2
    codebook_MB = codebook_overhead_MB
    larc3_MB = param_count(larc3) * 2 / 1024**2
    larc2_MB = param_count(larc2) * 2 / 1024**2

    modes = [
        ("K3V3", 3, 3, larc3_MB),
        ("K2V2", 2, 2, larc2_MB),
        ("K3V2_optional", 3, 2, larc2_MB),
    ]

    rows = []
    for mode, kb, vb, larc_MB in modes:
        bits = kv_payload_effective_bits(kb, vb)
        for T in context_lengths:
            total_kv_scalars = batch_size * L * 2 * Hkv * T * D
            original_fp16_KV_MB = total_kv_scalars * 16 / 8 / 1024**2
            payload_MB = total_kv_scalars * bits / 8 / 1024**2

            rows.append({
                "mode": mode,
                "K_bits": kb,
                "V_bits": vb,
                "T_context": T,
                "batch_size": batch_size,
                "payload_effective_bits": bits,
                "payload_compression_vs_fp16": 16 / bits,
                "original_fp16_KV_MB": original_fp16_KV_MB,
                "KV_payload_MB": payload_MB,
                "fixed_sigma_MB": sigma_MB,
                "fixed_codebook_MB": codebook_MB,
                "LARC_fixed_MB_fp16": larc_MB,
                "total_fixed_overhead_MB": sigma_MB + codebook_MB + larc_MB,
                "fixed_overhead_pct_of_KV_payload": 100.0 * (sigma_MB + codebook_MB + larc_MB) / max(payload_MB, 1e-9),
            })

    return pd.DataFrame(rows)

eff_df = specialized_effective_bits_report(model)
display(eff_df)

print("="*80)
print("SUMMARY")
print("="*80)
print(f"Headwise fixed sigma overhead MB : {(sigma_k_lh.numel() + sigma_v_lh.numel()) * 2 / 1024**2:.6f}")
print(f"Learned codebook overhead MB     : {codebook_overhead_MB:.6f}")
print(f"LARC3 fixed overhead MB          : {param_count(larc3) * 2 / 1024**2:.6f}")
print(f"LARC2 fixed overhead MB          : {param_count(larc2) * 2 / 1024**2:.6f}")
print("")
for mode, kb, vb in [("K3V3",3,3), ("K2V2",2,2), ("K3V2",3,2)]:
    b = kv_payload_effective_bits(kb, vb)
    print(f"{mode}: payload={b:.3f} bits/scalar, compression={16/b:.3f}x")

## Next interpretation

성공 기준:

```text
3bit-only:
K3V3 + LARC3 PPL < previous mixed 3bit + LARC PPL

2bit-only:
K2V2 + LARC2 PPL < previous mixed 2bit + LARC PPL

Optional:
K3V2 + LARC2가 PPL < 2.0 근처이면 2.5bit 방향이 가장 유망
```

이 노트북은 다음 단계로 확장할 수 있습니다.

```text
1. 3bit-only에 kl_topk 512, train_steps 300
2. 2bit-only에 K3V2 curriculum pretraining
3. Score-LARC를 temperature보다 더 강한 low-rank score bias로 확장
4. int2/int3 packed KV + fused codebook attention kernel
```

# Add-on: Dedicated Dual-LARC for K3V2 and K2V2

이 섹션은 기존 Specialized TQ-LARC 노트북 뒤에 추가된 실험입니다.

목표:

```text
K3V2 전용:
K = 3bit, V = 2bit, payload = 2.5bit
Score-LARC + Output-LARC 전용 학습

K2V2 전용:
K = 2bit, V = 2bit, payload = 2.0bit
Score-LARC + Output-LARC 전용 학습
```

핵심 구조:

```text
Score-LARC:
attention score를 softmax 전에 보정

Output-LARC:
attention output을 o_proj 이후 residual로 보정
```

In [ ]:
# ============================================================
# CELL A1. Dual-LARC modules
# Score-LARC + Output-LARC
# ============================================================

import math
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display


class ScoreLARC(nn.Module):
    # Lightweight token/key-dependent score correction before softmax.
    def __init__(self, num_layers, num_heads, hidden_size, init_scale=0.01):
        super().__init__()
        self.num_layers = num_layers
        self.num_heads = num_heads

        # key_stat_weight maps [k_code_abs, k_saturation, k_quant_error, 1] -> per-head score bias
        self.key_stat_weight = nn.Parameter(torch.zeros(num_layers, num_heads, 4))
        self.q_gate_proj = nn.Linear(hidden_size, num_heads, bias=True)
        self.score_log_temp = nn.Parameter(torch.zeros(num_layers, num_heads))

        self.logit_scale = nn.Parameter(
            torch.tensor(math.log(init_scale / (1.0 - init_scale)))
        )

        nn.init.zeros_(self.q_gate_proj.weight)
        nn.init.constant_(self.q_gate_proj.bias, -2.0)

    def forward(self, layer_idx, hidden_states, key_score_stats, attn_scores):
        # hidden_states: [B,Tq,C]
        # key_score_stats: [B,Tk,4]
        # attn_scores: [B,H,Tq,Tk]
        B, H, Tq, Tk = attn_scores.shape

        temp = torch.exp(self.score_log_temp[int(layer_idx)].clamp(-0.35, 0.35))
        attn_scores = attn_scores * temp.view(1, H, 1, 1).to(attn_scores.dtype)

        w = self.key_stat_weight[int(layer_idx)].to(attn_scores.dtype)  # [H,4]
        key_bias = torch.einsum(
            "bks,hs->bhk",
            key_score_stats.to(attn_scores.dtype),
            w,
        )  # [B,H,Tk]

        q_gate = torch.sigmoid(self.q_gate_proj(hidden_states)).transpose(1, 2)
        q_gate = q_gate.to(attn_scores.dtype)  # [B,H,Tq]

        scale = torch.sigmoid(self.logit_scale).to(attn_scores.dtype)
        score_delta = scale * q_gate.unsqueeze(-1) * key_bias.unsqueeze(2)

        return attn_scores + score_delta

    def reg_loss(self):
        return (
            (self.score_log_temp.float() ** 2).mean()
            + (self.key_stat_weight.float() ** 2).mean()
        )


class OutputMoBLARC(nn.Module):
    # Mixture-of-Basis output residual corrector.
    def __init__(
        self,
        num_layers,
        hidden_size,
        stat_dim,
        feature_dim=160,
        rank=128,
        experts=4,
        mlp_hidden=384,
        init_scale=0.01,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.rank = rank
        self.experts = experts

        self.hidden_proj = nn.Linear(hidden_size, feature_dim, bias=False)
        self.out_proj = nn.Linear(hidden_size, feature_dim, bias=False)
        self.stat_proj = nn.Linear(stat_dim, feature_dim, bias=True)
        self.layer_emb = nn.Embedding(num_layers, feature_dim)

        self.norm = nn.LayerNorm(feature_dim)

        self.coeff_net = nn.Sequential(
            nn.Linear(feature_dim, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, experts * rank),
        )

        self.expert_gate = nn.Sequential(
            nn.Linear(feature_dim, mlp_hidden // 2),
            nn.GELU(),
            nn.Linear(mlp_hidden // 2, experts),
        )

        self.token_gate = nn.Sequential(
            nn.Linear(feature_dim, mlp_hidden // 2),
            nn.GELU(),
            nn.Linear(mlp_hidden // 2, 1),
        )

        self.basis = nn.Parameter(torch.randn(experts, rank, hidden_size) * 0.01)
        self.logit_scale = nn.Parameter(
            torch.tensor(math.log(init_scale / (1.0 - init_scale)))
        )

        nn.init.zeros_(self.coeff_net[-1].weight)
        nn.init.zeros_(self.coeff_net[-1].bias)
        nn.init.zeros_(self.expert_gate[-1].weight)
        nn.init.zeros_(self.expert_gate[-1].bias)
        nn.init.zeros_(self.token_gate[-1].weight)
        nn.init.constant_(self.token_gate[-1].bias, -2.0)

    def forward_layer(self, layer_idx, hidden_states, base_out, quant_stats):
        B, T, C = hidden_states.shape
        layer_ids = torch.full(
            (B, T),
            int(layer_idx),
            device=hidden_states.device,
            dtype=torch.long,
        )

        feat = (
            self.hidden_proj(hidden_states)
            + self.out_proj(base_out)
            + self.stat_proj(quant_stats.to(base_out.dtype))
            + self.layer_emb(layer_ids).to(base_out.dtype)
        )

        feat = self.norm(feat)

        coeff = self.coeff_net(feat).view(B, T, self.experts, self.rank)
        egate = F.softmax(self.expert_gate(feat).float(), dim=-1).to(feat.dtype)

        delta_e = torch.einsum("bter,erh->bteh", coeff, self.basis.to(coeff.dtype))
        delta = (delta_e * egate.unsqueeze(-1)).sum(dim=2)

        tgate = torch.sigmoid(self.token_gate(feat))
        gscale = torch.sigmoid(self.logit_scale).to(delta.dtype)

        correction = gscale * tgate * delta
        return base_out + correction, correction, tgate.mean()


class DualLARC(nn.Module):
    def __init__(
        self,
        num_layers,
        num_heads,
        hidden_size,
        stat_dim,
        feature_dim=160,
        rank=128,
        experts=4,
        mlp_hidden=384,
        init_scale=0.01,
    ):
        super().__init__()
        self.score_larc = ScoreLARC(
            num_layers=num_layers,
            num_heads=num_heads,
            hidden_size=hidden_size,
            init_scale=init_scale,
        )
        self.output_larc = OutputMoBLARC(
            num_layers=num_layers,
            hidden_size=hidden_size,
            stat_dim=stat_dim,
            feature_dim=feature_dim,
            rank=rank,
            experts=experts,
            mlp_hidden=mlp_hidden,
            init_scale=init_scale,
        )

    def apply_score(self, layer_idx, hidden_states, key_score_stats, attn_scores):
        return self.score_larc(layer_idx, hidden_states, key_score_stats, attn_scores)

    def apply_output(self, layer_idx, hidden_states, base_out, quant_stats):
        return self.output_larc.forward_layer(layer_idx, hidden_states, base_out, quant_stats)

    def reg_loss(self):
        return self.score_larc.reg_loss()


def count_params_dual(module):
    return sum(p.numel() for p in module.parameters())

In [ ]:
# ============================================================
# CELL A2. Instantiate dedicated K3V2 and K2V2 Dual-LARC models
# ============================================================

larc_k3v2 = DualLARC(
    num_layers=num_layers,
    num_heads=num_q_heads,
    hidden_size=hidden_size,
    stat_dim=STAT_DIM,
    feature_dim=160,
    rank=128,
    experts=4,
    mlp_hidden=384,
    init_scale=cfg.init_scale,
).to(device=device, dtype=torch_dtype)

larc_k2v2 = DualLARC(
    num_layers=num_layers,
    num_heads=num_q_heads,
    hidden_size=hidden_size,
    stat_dim=STAT_DIM,
    feature_dim=192,
    rank=160,
    experts=4,
    mlp_hidden=448,
    init_scale=cfg.init_scale,
).to(device=device, dtype=torch_dtype)

DUAL_LARC_CONTROLLERS = {
    "k3v2": larc_k3v2,
    "k2v2": larc_k2v2,
}

print("K3V2 Dual-LARC params:", count_params_dual(larc_k3v2))
print("K3V2 Dual-LARC fp16 MB:", count_params_dual(larc_k3v2) * 2 / 1024**2)

print("K2V2 Dual-LARC params:", count_params_dual(larc_k2v2))
print("K2V2 Dual-LARC fp16 MB:", count_params_dual(larc_k2v2) * 2 / 1024**2)

print("Total Dual-LARC params:", count_params_dual(larc_k3v2) + count_params_dual(larc_k2v2))

In [ ]:
# ============================================================
# CELL A3. Repatch attention with Dual-LARC
# Score correction before softmax + Output correction after o_proj
# ============================================================

from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv
import types


DUAL_STATE = {
    "quant_enabled": False,
    "larc_enabled": False,
    "k_bits": 3,
    "v_bits": 2,
    "active_dual": "k3v2",
    "collect_stats": False,
    "gate_means": [],
    "corr_norms": [],
}


def _get_layer_idx_dual(attn):
    if hasattr(attn, "layer_idx") and attn.layer_idx is not None:
        return int(attn.layer_idx)
    return int(getattr(attn, "_larc_layer_idx"))


def make_dual_quant_stats(layer_idx, bsz, q_len, key_len, k_stats, v_stats, dtype, device):
    k_abs, k_sat, k_err = k_stats
    v_abs, v_sat, v_err = v_stats

    if k_abs.shape[1] != q_len:
        k_abs_q = k_abs[:, -q_len:]
        k_sat_q = k_sat[:, -q_len:]
        k_err_q = k_err[:, -q_len:]
        v_abs_q = v_abs[:, -q_len:]
        v_sat_q = v_sat[:, -q_len:]
        v_err_q = v_err[:, -q_len:]
    else:
        k_abs_q, k_sat_q, k_err_q = k_abs, k_sat, k_err
        v_abs_q, v_sat_q, v_err_q = v_abs, v_sat, v_err

    sk_log = sigma_k_lh[int(layer_idx)].mean().to(device=device).float().log().expand(bsz, q_len)
    sv_log = sigma_v_lh[int(layer_idx)].mean().to(device=device).float().log().expand(bsz, q_len)
    depth = torch.full((bsz, q_len), float(layer_idx) / max(1, num_layers - 1), device=device)
    ones = torch.ones((bsz, q_len), device=device)

    stats = torch.stack(
        [
            sk_log,
            sv_log,
            k_abs_q.float(),
            k_sat_q.float(),
            k_err_q.float(),
            v_abs_q.float(),
            v_sat_q.float(),
            v_err_q.float(),
            depth,
            ones,
        ],
        dim=-1,
    )
    return stats.to(dtype=dtype)


def make_key_score_stats(k_abs, k_sat, k_err):
    ones = torch.ones_like(k_abs)
    return torch.stack(
        [
            k_abs.float(),
            k_sat.float(),
            k_err.float(),
            ones.float(),
        ],
        dim=-1,
    )


def make_dual_larc_attention_forward(attn_module):
    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        **kwargs,
    ):
        bsz, q_len, _ = hidden_states.size()
        layer_idx = _get_layer_idx_dual(self)

        k_bits = int(DUAL_STATE["k_bits"])
        v_bits = int(DUAL_STATE["v_bits"])

        num_heads = self.config.num_attention_heads
        num_kv_heads_local = self.config.num_key_value_heads
        head_dim_local = self.config.hidden_size // self.config.num_attention_heads

        query_states = self.q_proj(hidden_states).view(
            bsz, q_len, num_heads, head_dim_local
        ).transpose(1, 2)

        key_states = self.k_proj(hidden_states).view(
            bsz, q_len, num_kv_heads_local, head_dim_local
        ).transpose(1, 2)

        value_states = self.v_proj(hidden_states).view(
            bsz, q_len, num_kv_heads_local, head_dim_local
        ).transpose(1, 2)

        if position_embeddings is None:
            if position_ids is None:
                position_ids = torch.arange(q_len, device=hidden_states.device).unsqueeze(0)
            cos, sin = self.rotary_emb(value_states, position_ids)
        else:
            cos, sin = position_embeddings

        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            key_states, value_states = past_key_value.update(
                key_states, value_states, layer_idx, cache_kwargs
            )

        key_len = key_states.shape[-2]

        if DUAL_STATE["quant_enabled"]:
            key_states, k_abs, k_sat, k_err = tq_quantize_dequant_with_stats(
                key_states,
                layer_idx,
                "k",
                bits=k_bits,
            )
            value_states, v_abs, v_sat, v_err = tq_quantize_dequant_with_stats(
                value_states,
                layer_idx,
                "v",
                bits=v_bits,
            )

            quant_stats = make_dual_quant_stats(
                layer_idx,
                bsz,
                q_len,
                key_len,
                (k_abs, k_sat, k_err),
                (v_abs, v_sat, v_err),
                hidden_states.dtype,
                hidden_states.device,
            )

            key_score_stats = make_key_score_stats(k_abs, k_sat, k_err).to(hidden_states.device)
        else:
            quant_stats = torch.zeros(
                bsz,
                q_len,
                STAT_DIM,
                device=hidden_states.device,
                dtype=hidden_states.dtype,
            )
            key_score_stats = torch.zeros(
                bsz,
                key_len,
                4,
                device=hidden_states.device,
                dtype=hidden_states.dtype,
            )

        key_states_rep = repeat_kv(key_states, self.num_key_value_groups)
        value_states_rep = repeat_kv(value_states, self.num_key_value_groups)

        attn_scores = torch.matmul(
            query_states,
            key_states_rep.transpose(2, 3),
        ) / math.sqrt(head_dim_local)

        if DUAL_STATE["quant_enabled"] and DUAL_STATE["larc_enabled"]:
            ctrl = DUAL_LARC_CONTROLLERS[DUAL_STATE["active_dual"]]
            attn_scores = ctrl.apply_score(
                layer_idx,
                hidden_states,
                key_score_stats,
                attn_scores,
            )

        if attention_mask is not None:
            causal_mask = attention_mask[:, :, :, : key_states_rep.shape[-2]]
            attn_scores = attn_scores + causal_mask

        attn_weights = F.softmax(attn_scores.float(), dim=-1).to(query_states.dtype)
        attn_weights = F.dropout(attn_weights, p=self.attention_dropout, training=self.training)

        attn_output = torch.matmul(attn_weights, value_states_rep)
        attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, -1)

        base_out = self.o_proj(attn_output)

        if DUAL_STATE["quant_enabled"] and DUAL_STATE["larc_enabled"]:
            ctrl = DUAL_LARC_CONTROLLERS[DUAL_STATE["active_dual"]]
            corrected, corr, gate_mean = ctrl.apply_output(
                layer_idx,
                hidden_states,
                base_out,
                quant_stats,
            )

            if DUAL_STATE["collect_stats"]:
                DUAL_STATE["gate_means"].append(float(gate_mean.detach().float().cpu()))
                DUAL_STATE["corr_norms"].append(
                    float((corr.float().norm() / (base_out.float().norm() + 1e-8)).detach().cpu())
                )

            attn_output_final = corrected
        else:
            attn_output_final = base_out

        if not output_attentions:
            attn_weights = None

        return attn_output_final, attn_weights

    return forward


def patch_dual_larc_attention():
    for i, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        attn._larc_layer_idx = i
        attn.forward = types.MethodType(make_dual_larc_attention_forward(attn), attn)
    print(f"Dual-LARC patched all {len(model.model.layers)} attention layers.")


patch_dual_larc_attention()

with torch.no_grad():
    DUAL_STATE["quant_enabled"] = False
    DUAL_STATE["larc_enabled"] = False
    y = model(**eval_batch0, use_cache=False).logits

print("smoke logits:", y.shape)

In [ ]:
# ============================================================
# CELL A4. Evaluation function for Dual-LARC
# ============================================================

@torch.no_grad()
def evaluate_dual_larc(
    label,
    quant=False,
    larc=False,
    k_bits=3,
    v_bits=2,
    active_dual="k3v2",
    n_batches=None,
):
    if n_batches is None:
        n_batches = cfg.eval_batches

    DUAL_STATE["quant_enabled"] = quant
    DUAL_STATE["larc_enabled"] = larc
    DUAL_STATE["k_bits"] = int(k_bits)
    DUAL_STATE["v_bits"] = int(v_bits)
    DUAL_STATE["active_dual"] = active_dual
    DUAL_STATE["collect_stats"] = False

    total_loss = 0.0
    total_tokens = 0
    elapsed = 0.0

    for bi in range(n_batches):
        batch, labels_ref = make_batch_from_ids(
            eval_ids_cpu,
            cfg.eval_batch_size,
            cfg.max_len,
            random_sample=False,
            batch_index=bi,
        )

        if device == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        out = model(**batch, use_cache=False)

        if device == "cuda":
            torch.cuda.synchronize()

        elapsed += time.perf_counter() - t0

        loss_sum, ntok = compute_nll_ppl_from_logits_and_labels(
            out.logits,
            labels_ref,
            batch.get("attention_mask"),
        )

        total_loss += loss_sum
        total_tokens += ntok

    nll = total_loss / max(total_tokens, 1)
    ppl = math.exp(nll)

    return {
        "label": label,
        "K_bits": k_bits if quant else 16,
        "V_bits": v_bits if quant else 16,
        "payload_eff_bits": kv_payload_effective_bits(k_bits, v_bits) if quant else 16.0,
        "nll": nll,
        "ppl": ppl,
        "loss_tokens": total_tokens,
        "elapsed_s": elapsed,
        "tokens_per_s": total_tokens / max(elapsed, 1e-9),
    }


dual_base_rows = [
    evaluate_dual_larc("fp_baseline", quant=False, larc=False),
    evaluate_dual_larc("dual_K3V2_base", quant=True, larc=False, k_bits=3, v_bits=2, active_dual="k3v2"),
    evaluate_dual_larc("dual_K2V2_base", quant=True, larc=False, k_bits=2, v_bits=2, active_dual="k2v2"),
]

dual_base_df = pd.DataFrame(dual_base_rows)
display(dual_base_df)

In [ ]:
# ============================================================
# CELL A5. Training function for dedicated Dual-LARC
# ============================================================

def autocast_ctx_dual():
    if device == "cuda":
        dtype = torch.bfloat16 if torch_dtype == torch.bfloat16 else torch.float16
        return torch.amp.autocast("cuda", enabled=cfg.amp, dtype=dtype)
    return torch.amp.autocast("cpu", enabled=False)


def teacher_topk_dual(logits, topk, temperature=1.0):
    vals, idx = torch.topk(logits.float(), k=topk, dim=-1)
    log_probs = F.log_softmax(vals / temperature, dim=-1)
    probs = log_probs.exp()
    return idx, probs, log_probs


def masked_topk_kl_from_teacher_dual(student_logits, top_idx, teacher_probs, teacher_log_probs, mask, temperature=1.0):
    student_log_probs_full = F.log_softmax(student_logits.float() / temperature, dim=-1)
    student_log_probs_topk = torch.gather(student_log_probs_full, dim=-1, index=top_idx)
    kl = (teacher_probs * (teacher_log_probs - student_log_probs_topk)).sum(dim=-1)
    return kl[mask].mean()


def masked_ce_from_logits_dual(student_logits, labels, mask):
    vocab = student_logits.size(-1)
    ce = F.cross_entropy(
        student_logits.float().reshape(-1, vocab),
        labels.reshape(-1),
        reduction="none",
    ).view_as(labels)
    return ce[mask].mean()


def train_dual_larc(
    active_dual,
    controller,
    k_bits,
    v_bits,
    steps=240,
    lr=1e-4,
    log_every=10,
):
    optimizer = torch.optim.AdamW(
        controller.parameters(),
        lr=lr,
        weight_decay=cfg.weight_decay,
    )

    history = []
    t0 = time.perf_counter()

    for step in range(1, steps + 1):
        train_batch, labels_ref = make_batch_from_ids(
            train_ids_cpu,
            cfg.train_batch_size,
            cfg.max_len,
            random_sample=True,
        )
        mask = train_batch["attention_mask"].bool()

        with torch.no_grad():
            DUAL_STATE["quant_enabled"] = False
            DUAL_STATE["larc_enabled"] = False

            teacher_logits = model(**train_batch, use_cache=False).logits.detach()
            top_idx, teacher_probs, teacher_log_probs = teacher_topk_dual(
                teacher_logits,
                cfg.kl_topk,
                cfg.kl_temperature,
            )

            DUAL_STATE["quant_enabled"] = True
            DUAL_STATE["larc_enabled"] = False
            DUAL_STATE["k_bits"] = int(k_bits)
            DUAL_STATE["v_bits"] = int(v_bits)
            DUAL_STATE["active_dual"] = active_dual

            base_quant_logits = model(**train_batch, use_cache=False).logits.detach()
            base_kl = masked_topk_kl_from_teacher_dual(
                base_quant_logits,
                top_idx,
                teacher_probs,
                teacher_log_probs,
                mask,
                cfg.kl_temperature,
            ).detach()

        optimizer.zero_grad(set_to_none=True)

        DUAL_STATE["quant_enabled"] = True
        DUAL_STATE["larc_enabled"] = True
        DUAL_STATE["k_bits"] = int(k_bits)
        DUAL_STATE["v_bits"] = int(v_bits)
        DUAL_STATE["active_dual"] = active_dual
        DUAL_STATE["collect_stats"] = True
        DUAL_STATE["gate_means"] = []
        DUAL_STATE["corr_norms"] = []

        with autocast_ctx_dual():
            student_logits = model(**train_batch, use_cache=False).logits

            kl = masked_topk_kl_from_teacher_dual(
                student_logits,
                top_idx,
                teacher_probs,
                teacher_log_probs,
                mask,
                cfg.kl_temperature,
            )

            ce = masked_ce_from_logits_dual(student_logits, labels_ref, mask)
            no_worse = F.relu(kl - base_kl)

            gate_mean = torch.tensor(
                sum(DUAL_STATE["gate_means"]) / max(1, len(DUAL_STATE["gate_means"])),
                device=kl.device,
            )
            corr_norm = torch.tensor(
                sum(DUAL_STATE["corr_norms"]) / max(1, len(DUAL_STATE["corr_norms"])),
                device=kl.device,
            )

            score_reg = controller.reg_loss()

            loss = (
                cfg.kl_weight * kl
                + cfg.ce_weight * ce
                + cfg.base_no_worse_weight * no_worse
                + cfg.residual_l2_weight * corr_norm
                + cfg.gate_l2_weight * gate_mean
                + 0.003 * score_reg
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(controller.parameters(), cfg.grad_clip)
        optimizer.step()

        if step == 1 or step % log_every == 0 or step == steps:
            row = {
                "active_dual": active_dual,
                "step": step,
                "K_bits": k_bits,
                "V_bits": v_bits,
                "payload_bits": kv_payload_effective_bits(k_bits, v_bits),
                "loss": float(loss.detach().cpu()),
                "kl_topk": float(kl.detach().cpu()),
                "base_kl_topk": float(base_kl.detach().cpu()),
                "ce_student": float(ce.detach().cpu()),
                "gate_mean": float(gate_mean.detach().cpu()),
                "corr_norm_ratio": float(corr_norm.detach().cpu()),
                "score_reg": float(score_reg.detach().cpu()),
                "elapsed_s": time.perf_counter() - t0,
            }
            history.append(row)
            print(row)

    return pd.DataFrame(history)

In [ ]:
# ============================================================
# CELL A6. Train dedicated K3V2 Dual-LARC
# Target: payload 2.5bit
# Previous target to beat: K3V2+LARC2 optional PPL ≈ 1.7935
# ============================================================

train_k3v2_dual_df = train_dual_larc(
    active_dual="k3v2",
    controller=larc_k3v2,
    k_bits=3,
    v_bits=2,
    steps=260,
    lr=1e-4,
    log_every=10,
)

display(train_k3v2_dual_df)

In [ ]:
# ============================================================
# CELL A7. Train dedicated K2V2 Dual-LARC
# Target: payload 2.0bit
# Previous target to beat: K2V2+LARC2 PPL ≈ 7.004
# ============================================================

train_k2v2_dual_df = train_dual_larc(
    active_dual="k2v2",
    controller=larc_k2v2,
    k_bits=2,
    v_bits=2,
    steps=300,
    lr=1e-4,
    log_every=10,
)

display(train_k2v2_dual_df)

In [ ]:
# ============================================================
# CELL A8. Final evaluation for Dual-LARC
# ============================================================

dual_final_rows = []

dual_final_rows.append(
    evaluate_dual_larc(
        "fp_baseline",
        quant=False,
        larc=False,
        n_batches=cfg.eval_batches,
    )
)

dual_final_rows.append(
    evaluate_dual_larc(
        "dual_K3V2_base",
        quant=True,
        larc=False,
        k_bits=3,
        v_bits=2,
        active_dual="k3v2",
        n_batches=cfg.eval_batches,
    )
)

dual_final_rows.append(
    evaluate_dual_larc(
        "dual_K3V2_ScoreOutput_LARC",
        quant=True,
        larc=True,
        k_bits=3,
        v_bits=2,
        active_dual="k3v2",
        n_batches=cfg.eval_batches,
    )
)

dual_final_rows.append(
    evaluate_dual_larc(
        "dual_K2V2_base",
        quant=True,
        larc=False,
        k_bits=2,
        v_bits=2,
        active_dual="k2v2",
        n_batches=cfg.eval_batches,
    )
)

dual_final_rows.append(
    evaluate_dual_larc(
        "dual_K2V2_ScoreOutput_LARC",
        quant=True,
        larc=True,
        k_bits=2,
        v_bits=2,
        active_dual="k2v2",
        n_batches=cfg.eval_batches,
    )
)

dual_final_df = pd.DataFrame(dual_final_rows)
display(dual_final_df)

print("=" * 80)
print("DUAL-LARC FINAL REPORT")
print("=" * 80)

fp = dual_final_df[dual_final_df["label"] == "fp_baseline"].iloc[0]
print(f"FP baseline PPL: {fp['ppl']:.6f}, NLL: {fp['nll']:.6f}")
print("")

def report_dual_pair(base_label, larc_label):
    base = dual_final_df[dual_final_df["label"] == base_label].iloc[0]
    larc = dual_final_df[dual_final_df["label"] == larc_label].iloc[0]

    print(f"[{larc_label}]")
    print(f"payload bits        : {larc['payload_eff_bits']:.3f}")
    print(f"base PPL            : {base['ppl']:.6f}")
    print(f"LARC PPL            : {larc['ppl']:.6f}")
    print(f"PPL improvement     : {base['ppl'] - larc['ppl']:+.6f}")
    print(f"base NLL            : {base['nll']:.6f}")
    print(f"LARC NLL            : {larc['nll']:.6f}")
    print(f"NLL improvement     : {base['nll'] - larc['nll']:+.6f}")
    print(f"compression vs fp16 : {16 / larc['payload_eff_bits']:.3f}x")

    if larc["ppl"] < base["ppl"]:
        print("✅ SUCCESS: Score+Output LARC improved quantized baseline.")
    else:
        print("❌ FAIL: Score+Output LARC did not improve quantized baseline.")

    if larc["ppl"] <= fp["ppl"] * 1.20:
        print("✅ Within 20% of FP baseline PPL.")
    else:
        print("⚠️ More than 20% worse than FP baseline PPL.")

    print("")

report_dual_pair("dual_K3V2_base", "dual_K3V2_ScoreOutput_LARC")
report_dual_pair("dual_K2V2_base", "dual_K2V2_ScoreOutput_LARC")

In [ ]:
# ============================================================
# CELL A9. Fixed overhead report
# LARC params are fixed model overhead, not KV-cache effective bits.
# ============================================================

dual_overhead_rows = []

for mode, kb, vb, ctrl in [
    ("K3V2_Dual", 3, 2, larc_k3v2),
    ("K2V2_Dual", 2, 2, larc_k2v2),
]:
    bits = kv_payload_effective_bits(kb, vb)

    for T in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
        total_kv_scalars = 1 * num_layers * 2 * num_kv_heads * T * head_dim
        fp16_kv_mb = total_kv_scalars * 16 / 8 / 1024**2
        payload_mb = total_kv_scalars * bits / 8 / 1024**2

        fixed_sigma_mb = (sigma_k_lh.numel() + sigma_v_lh.numel()) * 2 / 1024**2
        fixed_codebook_mb = codebook_overhead_MB
        larc_mb = count_params_dual(ctrl) * 2 / 1024**2
        fixed_total = fixed_sigma_mb + fixed_codebook_mb + larc_mb

        dual_overhead_rows.append({
            "mode": mode,
            "K_bits": kb,
            "V_bits": vb,
            "T_context": T,
            "payload_effective_bits": bits,
            "payload_compression_vs_fp16": 16 / bits,
            "original_fp16_KV_MB": fp16_kv_mb,
            "KV_payload_MB": payload_mb,
            "fixed_sigma_MB": fixed_sigma_mb,
            "fixed_codebook_MB": fixed_codebook_mb,
            "Dual_LARC_fixed_MB_fp16": larc_mb,
            "total_fixed_overhead_MB": fixed_total,
            "fixed_overhead_pct_of_KV_payload": 100.0 * fixed_total / max(payload_mb, 1e-9),
        })

dual_overhead_df = pd.DataFrame(dual_overhead_rows)
display(dual_overhead_df)

print("=" * 80)
print("DUAL-LARC OVERHEAD SUMMARY")
print("=" * 80)
print(f"K3V2 Dual-LARC params: {count_params_dual(larc_k3v2):,}")
print(f"K3V2 Dual-LARC fp16 MB: {count_params_dual(larc_k3v2) * 2 / 1024**2:.4f}")
print(f"K2V2 Dual-LARC params: {count_params_dual(larc_k2v2):,}")
print(f"K2V2 Dual-LARC fp16 MB: {count_params_dual(larc_k2v2) * 2 / 1024**2:.4f}")
print("")
print("Main KV-cache effective bits should be reported without fixed LARC params:")
print("K3V2 payload effective bits = 2.5 bits/scalar")
print("K2V2 payload effective bits = 2.0 bits/scalar")

## Expected targets

기존 결과 기준:

```text
K3V2 + LARC2 optional:
PPL ≈ 1.7935

K2V2 + LARC2 only:
PPL ≈ 7.0042
```

이번 Dual-LARC 목표:

```text
K3V2 Score+Output LARC:
PPL 1.65~1.75 목표

K2V2 Score+Output LARC:
PPL 5 이하 목표
```